# MRIscanner v2: keep training the existing models on new, never-seen data

This does **not** reset anything. It starts from the published checkpoints and trains further, on thousands of new images
the starting weights never saw, and keeps a model only if it beats the starting weights on validation data.

**New data added** (all downloaded automatically through your Kaggle login):
- BraTS 2020 *validation* patients (glioma; tumor slices picked from FLAIR, checked 96/96 against real masks)
- OASIS-1 healthy subjects (notumor; skull-stripped like BraTS, so the model can't learn "no skull = glioma")
- PMRAM, a Bangladeshi hospital MRI set with all 4 classes (the first independent meningioma/pituitary data in the project)

**No test leakage, enforced in code before any training:** nothing in Validation/Testing/ExternalTesting/PMRAMTesting is byte-identical to,
a perceptual near-duplicate of, or from the same patient as anything in Training. Any overlap aborts the run. The test sets are frozen
in a sha256 manifest that is re-verified before benchmarking.

### Before you press Run all
1. Runtime -> Change runtime type -> **GPU** (T4 works; A100/L4 is faster).
2. Drag your `kaggle.json` into `/content/` (left sidebar folder icon).
3. Runtime -> **Run all**, approve the Google Drive prompt, then leave it.

Progress and checkpoints are saved to Google Drive (`MyDrive/mriscanner_v2`). If Colab disconnects, reconnect and **Run all again**:
it resumes from the last saved epoch. Training runs up to 300 epochs per model but stops early when validation stops improving,
so a typical run ends well before 300.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/mriscanner_v2"
import os
os.makedirs(WORK_DIR, exist_ok=True)

In [ ]:
import os, subprocess
REPO_DIR = "/content/MRIscanner"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "https://github.com/christiandrep7/MRIscanner.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
assert os.path.isdir("src") and os.path.exists("run_continue_training.py")
print("cwd:", os.getcwd())

In [ ]:
!grep -v -E "^(torch|torchvision)==" requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt

In [ ]:
import os, shutil
from pathlib import Path
os.environ.pop("KAGGLE_API_TOKEN", None)
kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
src = next((c for c in ["/content/kaggle.json", "kaggle.json"] if os.path.exists(c)), None)
if src is None:
    raise FileNotFoundError("Drag kaggle.json into /content/ via the left sidebar, then Run all again.")
shutil.copy(src, kdir / "kaggle.json"); os.chmod(kdir / "kaggle.json", 0o600)
print("Kaggle credentials installed.")

### Build the data, run the leakage gate, train, benchmark
One command, fully resumable. Re-run this cell after any disconnect.
`--resume-every 3` writes the (large) resume state to Drive every 3 epochs instead of every epoch.

In [ ]:
!python -u run_continue_training.py --work-dir {WORK_DIR} --epochs 300 --patience 25 --resume-every 3

### Results: before (published checkpoints) vs after (continued training), on every frozen test set

In [ ]:
import json
from pathlib import Path
s = json.load(open(Path(WORK_DIR) / "outputs_v2/benchmark_v2/summary.json"))
sets = ["Testing", "ExternalTesting", "PMRAMTesting"]

def cell(t, arch):
    before, after = s.get(f"{t}/before/{arch}"), s.get(f"{t}/after/{arch}")
    if not (before and after):
        return "n/a"
    return f"{before['accuracy']:.3f} -> {after['accuracy']:.3f}"

print(f"{'accuracy before -> after':26s}" + "".join(f"{t:>24s}" for t in sets))
for arch in ["resnet50", "efficientnet_b0", "vgg16"]:
    print(f"{arch:26s}" + "".join(f"{cell(t, arch):>24s}" for t in sets))


Checkpoints are in `MyDrive/mriscanner_v2/checkpoints_v2/`, benchmark reports and confusion matrices in `MyDrive/mriscanner_v2/outputs_v2/`.
Send those two folders back (or zip them below).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/mriscanner_v2_checkpoints", "zip", WORK_DIR, "checkpoints_v2")
shutil.make_archive("/content/mriscanner_v2_outputs", "zip", WORK_DIR, "outputs_v2")
files.download("/content/mriscanner_v2_checkpoints.zip")
files.download("/content/mriscanner_v2_outputs.zip")